# Track 4 — Recognize Natural Scenes

> **REFERENCE EXAMPLE — DO NOT SUBMIT UNCHANGED**
>
> This completed notebook demonstrates one reusable workflow. Run it to understand the decisions, then choose **a different dataset** for your final project and update the target or output, features, preprocessing, split, evaluation, and explanation to fit your data.


**Question:** Does an image show buildings, forest, glacier, mountain, sea, or street?

This intermediate reference uses the Intel Image Classification dataset from Kaggle. It demonstrates folder-based labels, train/validation separation, augmentation, transfer learning, and multiclass evaluation.

**A GPU is recommended.** In Colab, choose **Runtime → Change runtime type → T4 GPU** before training. The Kaggle download is approximately 350 MB, so this track takes longer to start than the other three.

Dataset: https://www.kaggle.com/datasets/puneet6060/intel-image-classification


In [ ]:
%pip -q install "kagglehub>=1.0.3"


In [ ]:
from pathlib import Path
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from tensorflow.keras import Sequential
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, Input, RandomFlip, RandomRotation, RandomZoom

RANDOM_STATE = 42
tf.keras.utils.set_random_seed(RANDOM_STATE)

dataset_dir = Path(kagglehub.dataset_download("puneet6060/intel-image-classification"))
train_dir = next(path for path in dataset_dir.rglob("seg_train") if (path / "buildings").exists())
test_dir = next(path for path in dataset_dir.rglob("seg_test") if (path / "buildings").exists())

print("train:", train_dir)
print("test: ", test_dir)


## 1. Inspect folders and class balance

For image datasets, folder names often supply the labels. Count files before training so missing or imbalanced classes do not remain hidden.


In [ ]:
class_counts = {
    class_dir.name: len(list(class_dir.glob("*")))
    for class_dir in sorted(train_dir.iterdir())
    if class_dir.is_dir()
}
display(pd.Series(class_counts, name="images").to_frame())

majority_baseline = max(class_counts.values()) / sum(class_counts.values())
print("majority-class baseline accuracy:", round(majority_baseline, 3))


## 2. Create train, validation, and test pipelines

The provided test folder remains untouched until final evaluation. A fixed 20% of the training folder becomes validation data. Every image is resized to the same shape, and batches are prefetched for speed.


In [ ]:
IMAGE_SIZE = (160, 160)
BATCH_SIZE = 32

train_data = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.20,
    subset="training",
    seed=RANDOM_STATE,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)
validation_data = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.20,
    subset="validation",
    seed=RANDOM_STATE,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)
test_data = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    shuffle=False,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

class_names = train_data.class_names
AUTOTUNE = tf.data.AUTOTUNE
train_data = train_data.prefetch(AUTOTUNE)
validation_data = validation_data.prefetch(AUTOTUNE)
test_data = test_data.prefetch(AUTOTUNE)
print(class_names)


In [ ]:
images, labels = next(iter(train_data))
plt.figure(figsize=(10, 7))
for index in range(12):
    plt.subplot(3, 4, index + 1)
    plt.imshow(images[index].numpy().astype("uint8"))
    plt.title(class_names[int(labels[index])])
    plt.axis("off")
plt.tight_layout()
plt.show()


## 3. Make the modeling decisions explicit

- **Augmentation** applies only during training and introduces small flips, rotations, and zooms.
- **MobileNetV2** supplies visual features learned from a large image dataset.
- **Frozen base model** means Week 8 trains only the small classification head. This is faster and reduces overfitting.
- **Preprocessing** uses the exact scaling MobileNetV2 expects.


In [ ]:
augmentation = Sequential([
    RandomFlip("horizontal"),
    RandomRotation(0.05),
    RandomZoom(0.10),
], name="augmentation")

base_model = MobileNetV2(
    input_shape=(*IMAGE_SIZE, 3),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False

model = Sequential([
    Input(shape=(*IMAGE_SIZE, 3)),
    augmentation,
    tf.keras.layers.Lambda(preprocess_input),
    base_model,
    GlobalAveragePooling2D(),
    Dropout(0.25),
    Dense(len(class_names), activation="softmax"),
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()


## 4. Train the reference model

Three epochs are enough for the first visible result. More training is not automatically better; compare training and validation curves for signs of overfitting.


In [ ]:
history = model.fit(
    train_data,
    validation_data=validation_data,
    epochs=3,
)

pd.DataFrame(history.history)[["accuracy", "val_accuracy"]].plot(
    figsize=(8, 4), marker="o", title="Training and validation accuracy"
)
plt.ylim(0, 1)
plt.show()


## 5. Evaluate by class

Overall accuracy can hide a weak class. Use the untouched test folder, a per-class report, and a confusion matrix.


In [ ]:
test_loss, test_accuracy = model.evaluate(test_data, verbose=0)
probabilities = model.predict(test_data, verbose=0)
predictions = probabilities.argmax(axis=1)
actual = np.concatenate([labels.numpy() for _, labels in test_data])

print("majority baseline accuracy:", round(majority_baseline, 3))
print("model test accuracy:       ", round(test_accuracy, 3))
print(classification_report(actual, predictions, target_names=class_names, digits=3))

matrix = confusion_matrix(actual, predictions)
ConfusionMatrixDisplay(matrix, display_labels=class_names).plot(
    cmap="Blues", xticks_rotation=45, values_format="d"
)
plt.show()


## 6. Inspect predictions

Look for both correct and incorrect examples. Ask whether the image itself is ambiguous, whether the label is questionable, or whether two classes share visual features.


In [ ]:
test_images, test_labels = next(iter(test_data))
test_probabilities = model.predict(test_images, verbose=0)
test_predictions = test_probabilities.argmax(axis=1)

plt.figure(figsize=(12, 9))
for index in range(12):
    plt.subplot(3, 4, index + 1)
    plt.imshow(test_images[index].numpy().astype("uint8"))
    predicted_name = class_names[int(test_predictions[index])]
    actual_name = class_names[int(test_labels[index])]
    color = "darkgreen" if predicted_name == actual_name else "crimson"
    plt.title(f"pred: {predicted_name}\ntrue: {actual_name}", color=color, fontsize=9)
    plt.axis("off")
plt.tight_layout()
plt.show()


## 7. Try one change

- remove augmentation and compare validation performance;
- change the image size or dropout rate;
- unfreeze only the final MobileNetV2 block and fine-tune with a very small learning rate;
- adapt the pipeline to plants, animals, foods, waste categories, or products;
- collect ten of your own images and describe where the model fails.

Change one decision at a time and keep the original result for comparison.


In [ ]:
# OPTIONAL fine-tuning starting point. Run only after the frozen model.
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

# TODO: train for 1–2 more epochs, then compare test accuracy and per-class errors.


## Required next step: choose your own dataset

The dataset in this notebook is only the completed example. For your final project, find another folder-based image dataset such as plants, animals, food, waste, weather, or manufactured objects.

Before changing code, write down:

1. the new dataset source and license;
2. what one row or example represents;
3. the new target or desired output;
4. possible leakage or unavailable information;
5. the split strategy and why it matches real use;
6. the preprocessing steps that must change;
7. the baseline and main metric; and
8. one visible result you will produce with your data.

You may reuse the overall workflow and model as a starting point. You may not use the unchanged reference notebook as the final project.


## Completion checklist

- [ ] I counted the class folders before training.
- [ ] I kept test images separate from training and validation.
- [ ] I can explain resizing, preprocessing, and augmentation.
- [ ] I compared with a majority-class baseline.
- [ ] I inspected per-class metrics and the confusion matrix.
- [ ] I reviewed correct and incorrect example images.
- [ ] I changed one decision and compared the result.
